# NOVA RETAIL — Unified Risk Matrix

## Consolidación Forense de Riesgo por Tienda
**Proyecto:** Diagnóstico de Prevención de Pérdidas  
**Autor:** Denz One  
**Fase:** Integración de señales operativas y económicas en una matriz priorizada  
**Objetivo:** Unificar los principales hallazgos del caso en una sola tabla de riesgo por tienda para priorizar intervención, escalamiento e investigación focalizada.

---

### Propósito de este notebook
Este notebook consolida en una sola vista analítica las señales más relevantes identificadas en fases anteriores:

- severidad de discrepancia entre despacho y recepción
- recepciones en horario atípico (05:00)
- exposición económica a Ghost SKUs
- concentración por ruta logística
- línea de mando regional
- opacidad estructural (`ghost_store`)

### Preguntas clave
1. ¿Qué tiendas combinan múltiples capas de riesgo?
2. ¿Qué señales pesan más en la priorización?
3. ¿Qué tan concentrado está el riesgo económico y operativo?
4. ¿Cómo traducimos hallazgos dispersos en una decisión ejecutiva única?

### Nota del analista
La matriz de riesgo no busca declarar culpabilidad.  
Busca priorizar de forma objetiva los nodos donde convergen vulnerabilidad estructural, pérdida material y comportamiento operativo atípico.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

DATA_PATH = Path("../data/raw")

stores = pd.read_csv(DATA_PATH / "stores.csv")
products_sap = pd.read_csv(DATA_PATH / "products_sap.csv")
dispatches = pd.read_csv(DATA_PATH / "dispatches.csv")
receptions = pd.read_csv(DATA_PATH / "receptions.csv")
products_as400 = pd.read_csv(DATA_PATH / "products_as400.csv")

print("Datasets cargados correctamente.")

In [ ]:
high_value_categories = ["ELECTRONICA", "TELEFONIA", "COMPUTO", "ELECTRODOMESTICOS"]

products_priority = products_sap[
    products_sap["category_sap"].isin(high_value_categories)
].copy()

products_priority = products_priority[
    products_priority["price_sap"] >= products_priority["price_sap"].quantile(0.95)
].copy()

print("SKUs prioritarios:", len(products_priority))

In [ ]:
dispatches_priority = dispatches.merge(
    products_priority[["sku_sap", "category_sap", "price_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="inner"
)

receptions_priority = receptions.merge(
    products_priority[["sku_sap", "category_sap", "price_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="inner"
)

merged_priority = dispatches_priority.merge(
    receptions_priority,
    left_on="dispatch_id",
    right_on="dispatch_reference",
    how="inner",
    suffixes=("_dispatch", "_reception")
)

print("Merged priority:", merged_priority.shape)

In [ ]:
merged_priority["reception_hour"] = (
    pd.to_datetime(merged_priority["reception_time"], format="%H:%M", errors="coerce")
    .dt.hour
)

merged_priority_units = merged_priority[
    merged_priority["unit_type"] == "UNIDAD"
].copy()

print("Merged priority units:", merged_priority_units.shape)

In [ ]:
severity_by_store = (
    merged_priority_units
    .groupby("store_id_dispatch")[["quantity_dispatched", "quantity_received"]]
    .sum()
    .reset_index()
)

severity_by_store["quantity_diff"] = (
    severity_by_store["quantity_dispatched"] - severity_by_store["quantity_received"]
)

severity_by_store["pct_diff"] = np.where(
    severity_by_store["quantity_dispatched"] > 0,
    (severity_by_store["quantity_diff"] / severity_by_store["quantity_dispatched"]) * 100,
    np.nan
)

severity_by_store = severity_by_store.sort_values("pct_diff", ascending=False)

severity_by_store.head(15)

In [ ]:
early_hour_counts = (
    merged_priority_units[merged_priority_units["reception_hour"] == 5]
    .groupby("store_id_dispatch")
    .size()
    .reset_index(name="recepciones_5am")
)

early_hour_counts.sort_values("recepciones_5am", ascending=False).head(15)

In [ ]:
apple_ghosts = products_as400[products_as400["is_ghost"] == True].copy()
ghost_sap_refs = apple_ghosts["sku_sap_reference"].dropna().tolist()

ghost_dispatches = dispatches[
    dispatches["sku"].isin(ghost_sap_refs)
].copy()

ghost_dispatch_value = ghost_dispatches.merge(
    products_sap[["sku_sap", "price_sap", "description_sap", "brand", "category_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="left"
)

ghost_dispatch_value["dispatch_value_mxn"] = (
    ghost_dispatch_value["quantity_dispatched"] * ghost_dispatch_value["price_sap"]
)

ghost_by_store = ghost_dispatch_value.groupby("store_id").agg(
    despachos_ghost=("dispatch_id", "count"),
    unidades_ghost=("quantity_dispatched", "sum"),
    valor_ghost_mxn=("dispatch_value_mxn", "sum")
).reset_index()

ghost_by_store.sort_values("valor_ghost_mxn", ascending=False).head(15)

In [ ]:
store_meta = stores[[
    "store_id",
    "store_name",
    "city",
    "state",
    "system",
    "cedis_route",
    "regional_manager_id",
    "ghost_store"
]].copy()

store_meta.head()

In [ ]:
risk_matrix = store_meta.merge(
    severity_by_store,
    left_on="store_id",
    right_on="store_id_dispatch",
    how="left"
).merge(
    early_hour_counts,
    left_on="store_id",
    right_on="store_id_dispatch",
    how="left"
).merge(
    ghost_by_store,
    left_on="store_id",
    right_on="store_id",
    how="left"
)

risk_matrix = risk_matrix.drop(columns=[
    "store_id_dispatch_x",
    "store_id_dispatch_y"
], errors="ignore")

risk_matrix["recepciones_5am"] = risk_matrix["recepciones_5am"].fillna(0)
risk_matrix["despachos_ghost"] = risk_matrix["despachos_ghost"].fillna(0)
risk_matrix["unidades_ghost"] = risk_matrix["unidades_ghost"].fillna(0)
risk_matrix["valor_ghost_mxn"] = risk_matrix["valor_ghost_mxn"].fillna(0)
risk_matrix["quantity_dispatched"] = risk_matrix["quantity_dispatched"].fillna(0)
risk_matrix["quantity_received"] = risk_matrix["quantity_received"].fillna(0)
risk_matrix["quantity_diff"] = risk_matrix["quantity_diff"].fillna(0)
risk_matrix["pct_diff"] = risk_matrix["pct_diff"].fillna(0)

risk_matrix.head(20)

In [ ]:
print("Tiendas en la matriz:", len(risk_matrix))
print("Tiendas con severidad > 0:", (risk_matrix["pct_diff"] > 0).sum())
print("Tiendas con recepciones 5am:", (risk_matrix["recepciones_5am"] > 0).sum())
print("Tiendas con exposición ghost:", (risk_matrix["valor_ghost_mxn"] > 0).sum())

In [ ]:
risk_matrix["route_north_flag"] = np.where(risk_matrix["cedis_route"] == "NORTE", 1, 0)
risk_matrix["ghost_store_flag"] = np.where(risk_matrix["ghost_store"] == True, 1, 0)

risk_matrix["pct_diff_norm"] = risk_matrix["pct_diff"] / risk_matrix["pct_diff"].max()
risk_matrix["recepciones_5am_norm"] = risk_matrix["recepciones_5am"] / risk_matrix["recepciones_5am"].max()
risk_matrix["valor_ghost_norm"] = risk_matrix["valor_ghost_mxn"] / risk_matrix["valor_ghost_mxn"].max()

risk_matrix[[
    "store_id",
    "pct_diff",
    "pct_diff_norm",
    "recepciones_5am",
    "recepciones_5am_norm",
    "valor_ghost_mxn",
    "valor_ghost_norm",
    "route_north_flag",
    "ghost_store_flag"
]].head(20)

In [ ]:
risk_matrix["route_north_flag"] = np.where(risk_matrix["cedis_route"] == "NORTE", 1, 0)
risk_matrix["ghost_store_flag"] = np.where(risk_matrix["ghost_store"] == True, 1, 0)

risk_matrix["pct_diff_norm"] = risk_matrix["pct_diff"] / risk_matrix["pct_diff"].max()
risk_matrix["recepciones_5am_norm"] = risk_matrix["recepciones_5am"] / risk_matrix["recepciones_5am"].max()
risk_matrix["valor_ghost_norm"] = risk_matrix["valor_ghost_mxn"] / risk_matrix["valor_ghost_mxn"].max()

risk_matrix[[
    "store_id",
    "pct_diff",
    "pct_diff_norm",
    "recepciones_5am",
    "recepciones_5am_norm",
    "valor_ghost_mxn",
    "valor_ghost_norm",
    "route_north_flag",
    "ghost_store_flag"
]].head(20)

In [ ]:
risk_matrix["risk_score"] = (
    risk_matrix["pct_diff_norm"] * 0.35 +
    risk_matrix["recepciones_5am_norm"] * 0.25 +
    risk_matrix["valor_ghost_norm"] * 0.20 +
    risk_matrix["route_north_flag"] * 0.15 +
    risk_matrix["ghost_store_flag"] * 0.05
)

risk_matrix = risk_matrix.sort_values("risk_score", ascending=False)

risk_matrix[[
    "store_id",
    "store_name",
    "cedis_route",
    "regional_manager_id",
    "pct_diff",
    "recepciones_5am",
    "valor_ghost_mxn",
    "ghost_store",
    "risk_score"
]].head(20)

In [ ]:
risk_matrix["risk_score_v2"] = (
    risk_matrix["pct_diff_norm"] * 0.40 +
    risk_matrix["recepciones_5am_norm"] * 0.27 +
    risk_matrix["valor_ghost_norm"] * 0.20 +
    risk_matrix["route_north_flag"] * 0.08 +
    risk_matrix["ghost_store_flag"] * 0.05
)

risk_matrix = risk_matrix.sort_values("risk_score_v2", ascending=False)

risk_matrix[[
    "store_id",
    "store_name",
    "cedis_route",
    "regional_manager_id",
    "pct_diff",
    "recepciones_5am",
    "valor_ghost_mxn",
    "ghost_store",
    "risk_score",
    "risk_score_v2"
]].head(20)

In [ ]:
def assign_risk_tier(score):
    if score >= 0.60:
        return "TIER 1 - CRITICO"
    elif score >= 0.30:
        return "TIER 2 - ALTO"
    elif score > 0:
        return "TIER 3 - MONITOREO"
    else:
        return "TIER 4 - BASELINE"

risk_matrix["risk_tier"] = risk_matrix["risk_score_v2"].apply(assign_risk_tier)

risk_matrix[[
    "store_id",
    "store_name",
    "cedis_route",
    "pct_diff",
    "recepciones_5am",
    "valor_ghost_mxn",
    "ghost_store",
    "risk_score_v2",
    "risk_tier"
]].head(25)

In [ ]:
risk_matrix["risk_tier"].value_counts()

In [ ]:
def assign_risk_tier_v2(score):
    if score >= 0.65:
        return "TIER 1 - CRITICO"
    elif score >= 0.40:
        return "TIER 2 - ALTO"
    elif score >= 0.20:
        return "TIER 3 - MONITOREO"
    else:
        return "TIER 4 - BASELINE"

risk_matrix["risk_tier_v2"] = risk_matrix["risk_score_v2"].apply(assign_risk_tier_v2)

risk_matrix["risk_tier_v2"].value_counts()

In [ ]:
risk_matrix[[
    "store_id",
    "store_name",
    "cedis_route",
    "pct_diff",
    "recepciones_5am",
    "valor_ghost_mxn",
    "ghost_store",
    "risk_score_v2",
    "risk_tier_v2"
]].head(30)

In [ ]:
resumen_matriz_riesgo = pd.DataFrame({
    "indicador": [
        "Tiendas totales evaluadas",
        "Tiendas con score > 0",
        "Tiendas Tier 1 - Crítico",
        "Tiendas Tier 2 - Alto",
        "Tiendas Tier 3 - Monitoreo",
        "Tiendas Tier 4 - Baseline",
        "Ruta dominante en Tier 1",
        "Tiendas Tier 1 con recepciones 5 AM"
    ],
    "valor": [
        len(risk_matrix),
        int((risk_matrix["risk_score_v2"] > 0).sum()),
        int((risk_matrix["risk_tier_v2"] == "TIER 1 - CRITICO").sum()),
        int((risk_matrix["risk_tier_v2"] == "TIER 2 - ALTO").sum()),
        int((risk_matrix["risk_tier_v2"] == "TIER 3 - MONITOREO").sum()),
        int((risk_matrix["risk_tier_v2"] == "TIER 4 - BASELINE").sum()),
        risk_matrix[risk_matrix["risk_tier_v2"] == "TIER 1 - CRITICO"]["cedis_route"].mode()[0],
        int(((risk_matrix["risk_tier_v2"] == "TIER 1 - CRITICO") & (risk_matrix["recepciones_5am"] > 0)).sum())
    ]
})

resumen_matriz_riesgo

## Conclusiones de la matriz de riesgo unificada

### Hallazgos principales
- La matriz consolidó en una sola vista las principales señales del caso: severidad de discrepancia, recepciones 5 AM, exposición a Ghost SKUs, ruta logística y opacidad estructural.
- El score de riesgo permitió priorizar tiendas no solo por una señal aislada, sino por la convergencia de múltiples capas de vulnerabilidad.
- La versión final de tiers generó una estructura operativamente accionable:
  - un Tier 1 pequeño y quirúrgico para intervención inmediata
  - un Tier 2 ampliado para prevención reforzada
  - un Tier 3 monitoreable
  - y un baseline creíble para el resto de la red
- El Tier 1 concentra las tiendas donde la pérdida material, el horario atípico y la exposición operativa convergen con mayor fuerza.

### Interpretación metodológica
La matriz no sustituye una investigación formal, pero sí resuelve el problema más importante de priorización:
transformar múltiples hallazgos dispersos en una secuencia clara de acción ejecutiva.

### Decisión analítica
La compañía ya no necesita mirar toda la red con la misma intensidad.  
Ahora puede intervenir primero donde el riesgo es simultáneamente más severo, más visible y más económicamente relevante.

## Próximo paso recomendado

Con la matriz de riesgo consolidada, la siguiente etapa debe enfocarse en:

1. **Traducir esta priorización a una narrativa ejecutiva visual**
2. **Preparar visualizaciones premium para comité**
3. **Construir una presentación HTML interactiva con enfoque CEO / CFO / Legal**
4. **Definir recomendaciones diferenciadas por Tier de riesgo**

La fase analítica ya permitió responder la pregunta de priorización.  
La siguiente fase debe responder la pregunta de comunicación:
**cómo convertir evidencia compleja en decisión ejecutiva clara.**

In [ ]:
import json
import os

# Asegurar que la carpeta data existe en la raíz
if not os.path.exists('../data'):
    os.makedirs('../data')

# Calcular KPIs reales
kpis = {
    "total_risk_value": f"${risk_matrix['valor_ghost_mxn'].sum() / 1e6:.1f}M",
    "max_discrepancy": f"{risk_matrix['pct_diff'].max():.1f}%",
    "critical_stores": int((risk_matrix['risk_tier_v2'] == 'TIER 1 - CRITICO').sum()),
    "pattern_hour": "05:00 AM"
}

# Guardar en la raíz del proyecto para que GitHub Pages lo vea
with open('../data/kpis.json', 'w') as f:
    json.dump(kpis, f)

# Guardar el Top 10
risk_matrix.head(10).to_json('../data/top_risk.json', orient='records')

print("✅ Datos exportados con éxito a /data")
